In [10]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
import os
import json

# ==========================================
# 1. Configuration & Custom Layer Definition
# ==========================================
# Paths
data_dir = r"C:\AINutriCare\Data\Transformed"
model_path = r"C:\AINutriCare\Notebooks\Milestone_2\attention_lstm_87pct.h5"

# Feature Names (Must correspond to the columns in X_final.npy)
FEATURE_NAMES = [
    'Creatinine', 'Glucose', 'Lactate', 'Potassium', 'Sodium', 'pH', 
    'HeartRate', 'TotalInput', 'TotalOutput', 'FluidBalance', 
    'Vasopressors', 'Sedatives', 'Antibiotics', 'Insulin'
]

# --- CORRECTED CUSTOM LAYER ---
# This class definition is built to handle the specific config saved in your .h5 file
class SimpleAttention(Layer):
    def __init__(self, units=64, **kwargs):
        # We explicitly accept 'units' and 'kwargs'
        super(SimpleAttention, self).__init__(**kwargs)
        self.units = units

    def get_config(self):
        # Allow Keras to verify the config matches
        config = super(SimpleAttention, self).get_config()
        config.update({"units": self.units})
        return config

    def build(self, input_shape):
        self.W1 = self.add_weight(name='att_w1', shape=(input_shape[-1], self.units),
                                 initializer='glorot_uniform')
        self.W2 = self.add_weight(name='att_w2', shape=(self.units, 1),
                                 initializer='glorot_uniform')
        self.b1 = self.add_weight(name='att_b1', shape=(self.units,), initializer='zeros')
        super(SimpleAttention, self).build(input_shape)

    def call(self, x):
        h = tf.nn.tanh(tf.matmul(x, self.W1) + self.b1)
        e = tf.squeeze(tf.matmul(h, self.W2), -1)
        alpha = tf.nn.softmax(e)
        context = x * tf.expand_dims(alpha, -1)
        context = tf.reduce_sum(context, axis=1)
        return context, alpha

# ==========================================
# 2. Load Resources
# ==========================================
print(f"Step 1: Loading resources...")
print(f"Model Path: {model_path}")

try:
    # Load Data
    X = np.load(os.path.join(data_dir, 'X_final.npy'))
    y = np.load(os.path.join(data_dir, 'y_final.npy'))
    
    # Load Model
    model = load_model(model_path, custom_objects={'SimpleAttention': SimpleAttention})
    print("SUCCESS: Model and Data loaded.")
    
except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    print("Ensure the class definition matches exactly what was used during training.")
    exit()

# ==========================================
# 3. Clinical Logic (Risk & Signals)
# ==========================================
def get_risk_category(score):
    if score > 0.7:
        return "High", "CRITICAL"
    elif score > 0.3:
        return "Moderate", "STABLE"
    else:
        return "Low", "RECOVERING"

def extract_clinical_signals(patient_matrix):
    """
    Extracts the patient's status at the LAST HOUR (Hour 23).
    Note: Data is StandardScaled. Value > 1.0 means ~1 Std Dev above average.
    """
    last_hour_data = patient_matrix[-1, :] 
    signals = {}
    for i, feature in enumerate(FEATURE_NAMES):
        signals[feature] = last_hour_data[i]
    return signals

# ==========================================
# 4. Diet Rule Engine (The Heart of the Logic)
# ==========================================
def generate_diet_plan(risk_level, signals):
    plan = {
        "Risk Assessment": risk_level,
        "Dietary Strategy": {},
        "Specific Adjustments": []
    }

    # --- A. Base Strategy (Based on ML Risk Score) ---
    if risk_level == "High":
        plan["Dietary Strategy"] = {
            "Calories": "25-30 kcal/kg/day",
            "Protein": "1.5-2.0 g/kg/day (Prevent Muscle Wasting)",
            "Carbohydrates": "Controlled, low glycemic",
            "Fats": "Moderate, omega-3 enriched",
            "Fluids": "Restricted, monitor balance",
            "Route": "Enteral nutrition preferred"
        }
    elif risk_level == "Moderate":
        plan["Dietary Strategy"] = {
            "Calories": "25 kcal/kg/day",
            "Protein": "1.2-1.5 g/kg/day",
            "Carbohydrates": "Balanced",
            "Fats": "Moderate",
            "Fluids": "As tolerated",
            "Route": "Oral / Enteral"
        }
    else:
        plan["Dietary Strategy"] = {
            "Calories": "20-25 kcal/kg/day",
            "Protein": "1.0 g/kg/day",
            "Carbohydrates": "Normal",
            "Fats": "Normal",
            "Fluids": "Normal",
            "Route": "Oral"
        }

    # --- B. Specific Adjustments (Based on Lab Values) ---
    
    # 1. Hyperglycemia Rule
    if signals['Glucose'] > 1.0: 
        plan["Specific Adjustments"].append("High Glucose Detected: Switch to Diabetes-Specific Formula (Low Carb).")
    
    # 2. Renal/Kidney Rule
    if signals['Creatinine'] > 1.0 or signals['Potassium'] > 1.0:
        plan["Specific Adjustments"].append("Renal Stress Detected: Restrict Potassium (< 2g/day) and Phosphate.")

    # 3. Sepsis/Shock Rule
    if signals['Lactate'] > 1.0:
        plan["Specific Adjustments"].append("Elevated Lactate: Ensure Adequate Perfusion Support; High Protein Density.")
        
    # 4. Fluid Overload Rule
    if signals['FluidBalance'] > 1.0:
        plan["Specific Adjustments"].append("Positive Fluid Balance: Strict Fluid Restriction (1.5L max/day).")

    return plan

# ==========================================
# 5. Pipeline Execution Wrapper
# ==========================================
def run_clinical_pipeline(patient_idx):
    print(f"\n{'='*60}")
    print(f" PIPELINE REPORT: Patient Index {patient_idx}")
    print(f"{'='*60}")

    # 1. Get Patient Data
    patient_data = X[patient_idx] # Shape (24, 14)
    
    # 2. AI Prediction
    # Reshape to (1, 24, 14) for the LSTM model
    input_tensor = patient_data.reshape(1, 24, patient_data.shape[1])
    risk_prob = model.predict(input_tensor, verbose=0)[0][0]
    
    # 3. Clinical Context
    risk_level, status = get_risk_category(risk_prob)
    
    print(f"[AI Model Output]")
    print(f"Mortality Probability: {risk_prob:.2%}")
    print(f"Clinical Status: {status} ({risk_level} Risk)")

    # 4. Signal Extraction & Rule Engine
    signals = extract_clinical_signals(patient_data)
    diet_plan = generate_diet_plan(risk_level, signals)

    # 5. Output JSON Plan
    print(f"\n[Generated Clinical Diet Plan]")
    print(json.dumps(diet_plan, indent=4))

# ==========================================
# 6. Run Examples
# ==========================================

# Run for a standard patient (Index 0)
print("\n--- Example 1: Standard Case ---")
run_clinical_pipeline(patient_idx=5)

# Run for a Critical Patient (Label 1)
critical_indices = np.where(y == 1)[0]
if len(critical_indices) > 0:
    print("\n--- Example 2: Critical Case (Mortality Outcome) ---")
    run_clinical_pipeline(patient_idx=critical_indices[0])
else:
    print("\nNote: No critical patients found in this subset.")

Step 1: Loading resources...
Model Path: C:\AINutriCare\Notebooks\Milestone_2\attention_lstm_87pct.h5
SUCCESS: Model and Data loaded.

--- Example 1: Standard Case ---

 PIPELINE REPORT: Patient Index 5
[AI Model Output]
Mortality Probability: 36.96%
Clinical Status: STABLE (Moderate Risk)

[Generated Clinical Diet Plan]
{
    "Risk Assessment": "Moderate",
    "Dietary Strategy": {
        "Calories": "25 kcal/kg/day",
        "Protein": "1.2-1.5 g/kg/day",
        "Carbohydrates": "Balanced",
        "Fats": "Moderate",
        "Fluids": "As tolerated",
        "Route": "Oral / Enteral"
    },
    "Specific Adjustments": [
        "High Glucose Detected: Switch to Diabetes-Specific Formula (Low Carb).",
        "Renal Stress Detected: Restrict Potassium (< 2g/day) and Phosphate.",
        "Elevated Lactate: Ensure Adequate Perfusion Support; High Protein Density."
    ]
}

--- Example 2: Critical Case (Mortality Outcome) ---

 PIPELINE REPORT: Patient Index 18
[AI Model Output]
Morta